# FEATURES ENGiNEERING

## Initialisation spark et chargement data

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, schema_of_json
from pyspark.sql.types import StringType, IntegerType, DoubleType, ArrayType, StructType
import os
from pathlib import Path
import sys
#project_root = Path(__file__).parent.parent
current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.append(parent_dir)

#------------------------------
from notebooks import DataLoaderExploration,Config,DataPreprocessor,FeatureEngineer
    
#--------------

app_name="ArXivArticle-Classification"
master="spark://tawfekh-d:7077"#"spark://$(hostname):7077"  #"local[*]"   # Remplacer par cluster URL en production
memory= "4g"
executor_memory= "2g"
driver_memory= "2g"
# cores_per_executor: 4
# num_executors: 2
  
def _create_spark_session():
        """Créer une session Spark optimisée"""
        return SparkSession.builder \
            .appName("features_engineering") \
            .getOrCreate()
            # .master(master) \
            # .config('spark.driver.memory', memory) \
            # .config('spark.driver.executor.memory', executor_memory) \
            # .config('spark.sql.adaptive.enabled', 'true') \
            # .config('spark.sql.adaptive.coalescePartitions.enabled', 'true') \
            
spark=_create_spark_session()

loader=DataLoaderExploration(spark, Config)
dataprocess=DataPreprocessor(spark)
features=FeatureEngineer(spark)

26/01/08 03:41:04 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
df=loader.load_json_data("../data/processed/processed_sampled110k_arxiv_data_final.json")
loader.explore_data(df)

Chargement des données depuis ../data/processed/processed_sampled110k_arxiv_data_final.json...


 110,664 articles chargés

 =Exploration basique des données ===
Nombre de lignes: 110664
Nombre de colonnes: 10

Schéma:
root
 |-- abstract: string (nullable = true)
 |-- categories: string (nullable = true)
 |-- category_list: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- clean_abstract: string (nullable = true)
 |-- combined_text: string (nullable = true)
 |-- domain: string (nullable = true)
 |-- id: string (nullable = true)
 |-- main_category: string (nullable = true)
 |-- num_categories: long (nullable = true)
 |-- title: string (nullable = true)


Premières 5 lignes:
+--------------------------------------------------+-----------------------------------------------+--------------------------------------------------+--------------------------------------------------+--------------------------------------------------+--------+---------+---------------+--------------+--------------------------------------------------+
|                                

## tokenization + TF-IDF

In [3]:
# print("Chargement de la configuration")
# config = Config()

# 1. Pipeline de base (tokenization + TF-IDF)
pipeline = features.create_text_pipeline("combined_text")
model = pipeline.fit(df)
df = model.transform(df)

 Création du pipeline de traitement de texte


In [4]:

# 2. N-grammes (bigrammes)
df = features.create_ngram_features(df, "filtered_words", n=2)

 Création des 2-grammes


## statistiques textuelles 

In [5]:
# 3. Statistiques textuelles
df = features.create_text_statistics(df, "combined_text")


 Calcul des statistiques textuelles


## Combinaison FEATURES

In [6]:
# 4. Features de métadonnées
df = features.create_metadata_features(df)

# 5. Combiner toutes les features
df_features = features.combine_features(df)

print(" Feature engineering terminé")  

Création des features de métadonnées
⚙️ Combinaison des features
 Features à combiner: ['tfidf_features', 'text_length', 'word_count', 'avg_word_length', 'unique_ratio']
Type de vecteur correct (VectorUDT)
 Vecteur de features créé (5 dimensions)
 Feature engineering terminé


## Analyse des features
        


### Nombre de features

In [7]:
# Compter le nombre de features créées
feature_columns = [col for col in df_features.columns if 'feature' in col.lower()]
other_columns = [col for col in df_features.columns if 'feature' not in col.lower()]

print(f"Features créées: {len(feature_columns)} colonnes de features")
print(f"   Colonnes de features: {', '.join(feature_columns[:5])}...")
print(f"   Autres colonnes: {len(other_columns)} (id, texte, labels, etc.)")


Features créées: 4 colonnes de features
   Colonnes de features: raw_features, tfidf_features, raw_2gram_features, features...
   Autres colonnes: 20 (id, texte, labels, etc.)


### DIMENSIONS ET STATISTIQUE

In [8]:
from pyspark.sql import functions as F
# Afficher la dimension des features principales
text_col="combined_text"
if 'features' in df_features.columns:
    # Échantillonner un vecteur pour voir sa dimension
    sample = df_features.select('features').limit(1).collect()[0]['features']
    print(f"   Dimension du vecteur 'features': {len(sample)}")

# Statistiques sur les textes
if text_col in df_features.columns:
    stats = df_features.select(
        F.mean(F.length(F.col(text_col))).alias("longueur_moyenne"),
        F.stddev(F.length(F.col(text_col))).alias("ecart_type"),
        F.count("*").alias("total")
    ).collect()[0]
    
    print(f"\n Statistiques texte:")
    print(f"   Longueur moyenne: {stats['longueur_moyenne']:.0f} caractères")
    print(f"   Écart-type: {stats['ecart_type']:.0f} caractères")
    print(f"   Total articles: {stats['total']:,}")

# 7. Sauvegarder les données avec features
print("\n Étape 6: Sauvegarde des données avec features")

   Dimension du vecteur 'features': 5004


[Stage 13:=======>                                                  (1 + 7) / 8]


 Statistiques texte:
   Longueur moyenne: 953 caractères
   Écart-type: 419 caractères
   Total articles: 110,664

 Étape 6: Sauvegarde des données avec features


In [9]:
show_columns=["category_list","num_categories","main_category" ,"domain"]
show_columns2=["abstract","title","combined_text"]
show_columns3=["raw_features", "tfidf_features", "raw_2gram_features", "features"]
# show_columns4=["raw_features", "tfidf_features", "raw_2gram_features", "features"]

df_selected = df_features.select(*show_columns)
df_selected2 = df_features.select(*show_columns2)
df_selected3 = df_features.select(*show_columns3)
print(f"\nPremières 5 lignes:")
df_selected.show(5, truncate=50)
df_selected2.show(5, truncate=50)
df_selected3.show(5, truncate=50)


Premières 5 lignes:
+--------------------------------------------------+--------------+---------------+--------+
|                                     category_list|num_categories|  main_category|  domain|
+--------------------------------------------------+--------------+---------------+--------+
|                                        [astro-ph]|             1|       astro-ph|astro-ph|
|                                        [astro-ph]|             1|       astro-ph|astro-ph|
|                                         [math.NT]|             1|        math.NT|    math|
|[physics.flu-dyn, hep-th, math-ph, math.MP, qua...|             5|physics.flu-dyn| physics|
|                                        [astro-ph]|             1|       astro-ph|astro-ph|
+--------------------------------------------------+--------------+---------------+--------+
only showing top 5 rows

+--------------------------------------------------+--------------------------------------------------+--------------

### Show final data and saving

In [10]:
loader.explore_data(df_features)
# #sauvegarde
loader.save_as_single_json(df_features,"../data/processed/features_sampled110k_arxiv_data_final.json")
print(f" Dataset enregistrer")


 =Exploration basique des données ===


Nombre de lignes: 110664
Nombre de colonnes: 24

Schéma:
root
 |-- abstract: string (nullable = true)
 |-- categories: string (nullable = true)
 |-- category_list: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- clean_abstract: string (nullable = true)
 |-- combined_text: string (nullable = true)
 |-- domain: string (nullable = true)
 |-- id: string (nullable = true)
 |-- main_category: string (nullable = true)
 |-- num_categories: long (nullable = true)
 |-- title: string (nullable = true)
 |-- words: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- filtered_words: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- raw_features: vector (nullable = true)
 |-- tfidf_features: vector (nullable = true)
 |-- 2_grams: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- raw_2gram_features: vector (nullable = true)
 |-- 2gram_tfidf: vector (nullable = true)
 |-- text_length: integer (nu

Fichier sauvegardé : ../data/processed/features_sampled110k_arxiv_data_final.json
 Taille : 1484.29 MB
 Dataset enregistrer


In [11]:
# Nettoyage de la session Spark
spark.stop()